In [30]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import re

In [31]:
df = pd.read_csv('../data/used_cars.csv')
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [32]:
def basic_info(df: pd.DataFrame):
    number_of_row = df.shape[0]
    number_of_column = df.shape[1]
    duplicates = df.duplicated().sum()
    mem_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f'Number of rows: {number_of_row}')
    print(f'Number of columns: {number_of_column}')
    print(f'Duplicates: {duplicates}')
    print(f'Memory usage: {mem_mb:.2f} MB')
basic_info(df)

Number of rows: 4009
Number of columns: 12
Duplicates: 0
Memory usage: 2.58 MB


In [33]:
_BOOL_MAP = {
    "yes": 1, "no": 0, "true": 1, "false": 0,
    "y": 1, "n": 0, "1": 1, "0": 0,
}
_SYMBOL_PATTERN = re.compile(
    r"""
    ^\s*                    # leading whitespace
    [₹$€£¥₩%+\-]?          # optional currency or sign prefix
    [\s]?                   # optional space after symbol
    |                       # OR
    \s*                     # trailing whitespace
    (mi\.?|km\.?|kg\.?|lbs?\.?|mph|kph|hp|cc|°[cf]?|%|[₹$€£¥₩])
    \s*$                    # end
    """,
    re.IGNORECASE | re.VERBOSE,
)
_DATE_CHAR_PATTERN = re.compile(r"[/\-:]|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec", re.IGNORECASE)

In [34]:
def _strip_to_numeric(value: str) -> str:
    """Remove currency symbols, units, commas from a string value."""
    v = _SYMBOL_PATTERN.sub("", str(value))         # strip prefix/suffix symbols
    v = v.replace(",", "")                          # remove thousand separators
    return v.strip()

In [35]:
def _is_dirty_numeric(series: pd.Series, sample_size: int = 50) -> bool:
    """
    Returns True if the column looks like a number wearing a costume.
    Tests on a sample: strip symbols → try float cast → check success rate.
    """
    sample = series.dropna().head(sample_size)
    if len(sample) == 0:
        return False
    success = 0
    for val in sample:
        try:
            float(_strip_to_numeric(str(val)))
            success += 1
        except ValueError:
            pass
    return (success / len(sample)) >= 0.90   # 90% must parse cleanly

In [36]:
def _is_dirty_boolean(series: pd.Series) -> bool:
    """
    Returns True if all unique non-null values map to yes/no/true/false etc.
    """
    unique_vals = set(series.dropna().astype(str).str.lower().unique())
    return unique_vals.issubset(_BOOL_MAP.keys())

In [37]:
def _is_mostly_numeric(series: pd.Series, threshold: float = 0.85) -> bool:
    """
    Returns True if pd.to_numeric coerces successfully on >threshold fraction.
    Catches columns like ["35", "40", "N/A", "55"] — mostly numeric, few bad rows.
    """
    coerced = pd.to_numeric(series, errors="coerce")
    non_null_original = series.notna().sum()
    if non_null_original == 0:
        return False
    success_rate = coerced.notna().sum() / non_null_original
    return success_rate >= threshold

In [38]:
def _clean_numeric_column(series: pd.Series) -> pd.Series:
    """Strip symbols and cast to float. Unparseable → NaN."""
    def _convert(val):
        try:
            return float(_strip_to_numeric(str(val)))
        except ValueError:
            return np.nan

    return series.apply(_convert)

In [39]:
def _clean_boolean_column(series: pd.Series) -> pd.Series:
    """Map yes/no/true/false → 1/0. Unknowns → NaN."""
    return series.astype(str).str.lower().map(_BOOL_MAP)

In [40]:
def _is_datetime_like(series: pd.Series, sample_size: int = 50, threshold: float = 0.85) -> bool:
    """
    Returns True if the column looks like dates.
    Guards against pure numeric columns (e.g. "2021") being misread as dates
    by requiring date-separator characters or month names in the raw text.
    """
    sample = series.dropna().astype(str).head(sample_size)
    if sample.empty:
        return False

    # require at least one value to contain a date-like character/word
    has_date_chars = sample.str.contains(_DATE_CHAR_PATTERN, regex=True).mean()
    if has_date_chars < 0.5:
        return False

    parsed = pd.to_datetime(sample, errors="coerce", format="mixed")
    success_rate = parsed.notna().mean()
    return success_rate >= threshold


def _clean_datetime_column(series: pd.Series) -> pd.Series:
    """Parse to datetime. Unparseable → NaT."""
    return pd.to_datetime(series, errors="coerce", format="mixed")

In [41]:
def run_basic_cleaning(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Detects and fixes dirty columns automatically.

    Pass in the raw loaded DataFrame. Returns:
        - cleaned_df  : DataFrame with corrected dtypes
        - cleaning_report : dict logging what was changed and why
    """
    df = df.copy()
    report = {
        "converted_to_numeric": [],
        "converted_to_boolean": [],
        "coerced_mostly_numeric": [],
        "converted_to_datetime": [],
        "unchanged": [],
    }

    object_cols = df.select_dtypes(include=["object", "string", "category"]).columns

    for col in object_cols:
        series = df[col]

        # ── Priority 0: Datetime check (must run before boolean/numeric) ──────────
        if _is_datetime_like(series):
            df[col] = _clean_datetime_column(series)
            report["converted_to_datetime"].append(col)

        # ── Priority 1: Boolean check first (yes/no cols are also "numeric-ish") ──
        elif _is_dirty_boolean(series):
            df[col] = _clean_boolean_column(series)
            report["converted_to_boolean"].append(col)

        # ── Priority 2: Dirty numeric (currency, units, commas) ───────────────────
        elif _is_dirty_numeric(series):
            df[col] = _clean_numeric_column(series)
            report["converted_to_numeric"].append(col)

        # ── Priority 3: Mostly numeric with a few bad rows ────────────────────────
        elif _is_mostly_numeric(series):
            df[col] = pd.to_numeric(series, errors="coerce")
            report["coerced_mostly_numeric"].append(col)

        # ── No pattern matched: leave as-is for EDA to handle ────────────────────
        else:
            report["unchanged"].append(col)

    return df, report

In [52]:
cleaned_df, report = run_basic_cleaning(df)
cleaned_df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,51000.0,E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,1.0,10300.0
1,Hyundai,Palisade SEL,2021,34742.0,Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,1.0,38005.0
2,Lexus,RX 350 RX 350,2022,22372.0,Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,54598.0
3,INFINITI,Q50 Hybrid Sport,2015,88900.0,Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,1.0,15500.0
4,Audi,Q3 45 S line Premium Plus,2021,9835.0,Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,34999.0


In [47]:
print(report)

{'converted_to_numeric': ['milage', 'price'], 'converted_to_boolean': ['clean_title'], 'coerced_mostly_numeric': [], 'converted_to_datetime': [], 'unchanged': ['brand', 'model', 'fuel_type', 'engine', 'transmission', 'ext_col', 'int_col', 'accident']}


In [43]:
test_df = pd.DataFrame({
    "price": ["$10,300", "$38,005", "$54,598", "$15,500"],
    "milage": ["51,000 mi.", "34,742 mi.", "22,372 mi.", "88,900 mi."],
    "clean_title": ["Yes", "Yes", "No", "Yes"],
    "sale_date": ["2024-01-15", "15/02/2024", "Jan 20, 2024", "2024-03-10"],
    "model_year": [2013, 2021, 2022, 2015],   # already int — should stay unchanged
})
cleaned_df, report = run_basic_cleaning(test_df)
print(cleaned_df.dtypes)
print(report)

price                 float64
milage                float64
clean_title             int64
sale_date      datetime64[us]
model_year              int64
dtype: object
{'converted_to_numeric': ['price', 'milage'], 'converted_to_boolean': ['clean_title'], 'coerced_mostly_numeric': [], 'converted_to_datetime': ['sale_date'], 'unchanged': []}


In [53]:
cleaned_df.isnull().sum()

brand             0
model             0
model_year        0
milage            0
fuel_type       170
engine            0
transmission      0
ext_col           0
int_col           0
accident        113
clean_title     596
price             0
dtype: int64

In [74]:
## Missing Value report
def missing_value(df: pd.DataFrame):
    column = []
    for col in df:
        if df[col].isnull().any():
            print(col)
            print(df[col].isnull().sum())
            print(f"Missing Using Mean {df[col].isnull().mean() * 100:.2f}%")
            print(f"Missing Value {(df[col].isnull().sum() / len(df))*100:.2f}%")
    return column
print(missing_value(cleaned_df))

fuel_type
170
Missing Using Mean 4.24%
Missing Value 4.24%
accident
113
Missing Using Mean 2.82%
Missing Value 2.82%
clean_title
596
Missing Using Mean 14.87%
Missing Value 14.87%
[]
